# Herogod — Qwen Image Edit Studio (Colab)

An image-editing studio built on **Qwen-Image-Edit-2511** with the **MCNL v1** LoRA
(`ScottzillaSystems/qwen-image-edit-plus-nsfw-lora`). Fooocus-style styles, reference
images, true-CFG control, a modern before/after canvas, and a temporary public URL.

**Runtime → Change runtime type → GPU.**
* L4 / A100 (24–40 GB): the comfortable choice — 1024² edits in bf16.
* T4 (16 GB, free tier): works with fp16 and a smaller pixel budget; expect ~2–4 min per image.
* No GPU: it still runs, but a single image can take a very long time.

**Adults only.** These are open weights with an adult-content adapter. Never create sexual
content involving minors, non-consensual imagery, harassment or fraud. You are responsible
for what you generate and where you publish it.

Run the cells top to bottom. The last cell prints the studio URL and access token.


## 1 · Get the code

In [ ]:
import os
import subprocess
from pathlib import Path

# Set this if you forked the project.
REPO_URL = "https://github.com/tao-mihiranga/Herogod.git"
BRANCH = ""  # leave empty for the default branch

PROJECT_DIR = Path("/content/Herogod")
if (Path.cwd() / "server.py").is_file():
    PROJECT_DIR = Path.cwd()
elif not (PROJECT_DIR / "server.py").is_file():
    cmd = ["git", "clone", "--depth", "1"] + ([ "--branch", BRANCH] if BRANCH else []) + [REPO_URL, str(PROJECT_DIR)]
    subprocess.run(cmd, check=True)

os.chdir(PROJECT_DIR)
print("Project:", PROJECT_DIR)


## 2 · Runtime choices

In [ ]:
# @title Configuration { display-mode: "form" }
BASE_MODEL = "Qwen/Qwen-Image-Edit-2511" # @param ["Qwen/Qwen-Image-Edit-2511"] {allow-input: true}
LORA_REPO = "ScottzillaSystems/qwen-image-edit-plus-nsfw-lora" # @param {type:"string"}
DTYPE = "auto" # @param ["auto", "bf16", "fp16", "fp32"]
QUANT = "none" # @param ["none", "bnb4"]
OFFLOAD = "none" # @param ["none", "model", "sequential"]
MAX_PIXELS = 1048576 # @param {type:"integer"}
DEFAULT_STEPS = 20 # @param {type:"integer"}
DOWNLOAD_NOW = True # @param {type:"boolean"}
PUBLIC_TUNNEL = True # @param {type:"boolean"}

import os
import secrets

os.environ.update({
    "HEROGOD_BASE_MODEL": BASE_MODEL,
    "HEROGOD_LORA_REPO": LORA_REPO,
    "HEROGOD_DTYPE": DTYPE,
    "HEROGOD_QUANT": QUANT,
    "HEROGOD_OFFLOAD": OFFLOAD,
    "HEROGOD_MAX_PIXELS": str(MAX_PIXELS),
    "HEROGOD_STEPS": str(DEFAULT_STEPS),
    "HEROGOD_VAE_TILING": "1",
    "HEROGOD_MODELS_DIR": "/content/models",
    # A fresh token on every run. Paste it into the studio when prompted.
    "HEROGOD_API_TOKEN": secrets.token_urlsafe(24),
})
print("Configured. Access token generated (printed at the end).")

# Optional: add HF_TOKEN / HF_HUB_ENABLE_HF_TRANSFER in Colab Secrets (key icon, left bar).
try:
    from google.colab import userdata
    for name in ("HF_TOKEN", "HF_HUB_ENABLE_HF_TRANSFER"):
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = str(value)
    if os.environ.get("HF_TOKEN"):
        print("HF token loaded from Colab Secrets.")
except Exception:
    pass

try:
    import torch
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(f"GPU: {props.name} · {props.total_memory / 1024**3:.1f} GB VRAM")
        if props.total_memory < 20e9 and DTYPE == "auto":
            print("Tight VRAM detected: consider QUANT='bnb4' and MAX_PIXELS=786432.")
    else:
        print("No GPU attached. Edits will be extremely slow.")
except Exception:
    print("PyTorch will be installed in the next cell.")


## 3 · Install

In [ ]:
%%bash
set -e
echo "Installing server + diffusion stack (this is quiet but takes a few minutes)…"
pip -q install -r requirements.txt
pip -q install -r requirements-local.txt
if [ "$HEROGOD_QUANT" = "bnb4" ]; then
  pip -q install -r requirements-quant.txt
fi
python - <<'EOF'
import importlib, sys
for name in ("torch", "diffusers", "transformers", "accelerate", "fastapi"):
    module = importlib.import_module(name)
    print(f"{name:12} {getattr(module, '__version__', '?')}")
import torch
print("cuda:", torch.cuda.is_available())
EOF


## 4 · Download the weights into Colab

Qwen-Image-Edit-2511 is ~40 GB in bf16; the LoRA is 590 MB. They land in `/content/models`,
so the server never has to fetch them mid-request. Skip this cell if you prefer lazy loading
(the first generation will download them anyway, with less feedback).

Set `DOWNLOAD_NOW = False` above to skip.

In [ ]:
if DOWNLOAD_NOW:
    import subprocess, sys
    cmd = [sys.executable, "scripts/download_models.py", "--dest", "/content/models"]
    result = subprocess.run(cmd)
    if result.returncode != 0:
        print("\nDownload failed. Common causes: not enough disk space, network limits, or a gated repo.")
    else:
        import os
        base_dir = f"/content/models/{BASE_MODEL.replace('/', '--')}"
        if os.path.isdir(base_dir):
            os.environ["HEROGOD_BASE_MODEL_PATH"] = base_dir
        lora_dir = f"/content/models/{LORA_REPO.replace('/', '--')}"
        if os.path.isdir(lora_dir):
            os.environ["HEROGOD_LORA_REPO"] = lora_dir
        print("Model paths configured for local loading.")
else:
    print("Skipped. The model downloads on the first generation.")


## 5 · Start the studio server

In [ ]:
import socket
import subprocess
import sys
import time
from pathlib import Path

LOG = Path("/tmp/herogod-server.log")
server_log = LOG.open("w")
server_process = subprocess.Popen(
    [sys.executable, "server.py", "--host", "0.0.0.0", "--port", "8000"],
    stdout=server_log,
    stderr=subprocess.STDOUT,
    env={**__import__("os").environ},
)

for _ in range(60):
    if server_process.poll() is not None:
        raise RuntimeError("Server exited:\n" + LOG.read_text()[-4000:])
    try:
        with socket.create_connection(("127.0.0.1", 8000), timeout=1):
            break
    except OSError:
        time.sleep(0.5)
else:
    raise RuntimeError("Server did not open port 8000:\n" + LOG.read_text()[-4000:])

print("Herogod server is running on port 8000. Log:", LOG)


## 6 · Open the studio

Two ways to reach it:

* **Cloudflare quick tunnel** (works from any device, public URL, protected by the token).
* **Colab's own proxy** (only from this browser tab, no public exposure).

Set `PUBLIC_TUNNEL = False` in the configuration cell to use the Colab proxy instead.

In [ ]:
import os
import platform
import re
import stat
import subprocess
import time
import urllib.request
from pathlib import Path

TOKEN = os.environ["HEROGOD_API_TOKEN"]

if not PUBLIC_TUNNEL:
    from google.colab.output import eval_js
    url = eval_js("google.colab.kernel.proxyPort(8000)")
    print("Studio URL:", url)
    print("Access token:", TOKEN)
else:
    if platform.machine() not in ("x86_64", "AMD64"):
        raise RuntimeError("The cloudflared launcher expects a 64-bit Intel/AMD runtime. Use PUBLIC_TUNNEL=False.")
    cloudflared = Path("/tmp/cloudflared")
    if not cloudflared.exists():
        print("Downloading cloudflared…")
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            cloudflared,
        )
    cloudflared.chmod(cloudflared.stat().st_mode | stat.S_IEXEC)

    tunnel_log_path = Path("/tmp/herogod-tunnel.log")
    tunnel_log = tunnel_log_path.open("w")
    tunnel_process = subprocess.Popen(
        [str(cloudflared), "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
        stdout=tunnel_log,
        stderr=subprocess.STDOUT,
    )

    public_url = None
    for _ in range(120):
        if tunnel_process.poll() is not None:
            break
        match = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", tunnel_log_path.read_text())
        if match:
            public_url = match.group(0)
            break
        time.sleep(1)

    if not public_url:
        raise RuntimeError("Tunnel failed:\n" + tunnel_log_path.read_text()[-3000:])

    print("\n" + "─" * 72)
    print("HEROGOD IS READY")
    print("Studio:        ", public_url)
    print("One-click link:", f"{public_url}/?token={TOKEN}")
    print("Access token:  ", TOKEN)
    print("─" * 72)
    print("Open the studio, and if it asks for a token paste the one above.")
    print("Keep this notebook running; closing the runtime kills the tunnel and the model.")


## 7 · Health check

In [ ]:
import json
import urllib.request

request = urllib.request.Request(
    "http://127.0.0.1:8000/api/health",
)
with urllib.request.urlopen(request, timeout=30) as response:
    print(json.dumps(json.load(response), indent=2))

request = urllib.request.Request(
    "http://127.0.0.1:8000/api/engine",
    headers={"X-Herogod-Token": __import__("os").environ["HEROGOD_API_TOKEN"]},
)
with urllib.request.urlopen(request, timeout=120) as response:
    print(json.dumps(json.load(response), indent=2))


## Notes & troubleshooting

| Symptom | Fix |
| --- | --- |
| "Cannot reach the backend" | Re-run cell 5; check `/tmp/herogod-server.log`. |
| 401 token rejected | Copy the token printed in cell 6 into **Engine → Connection**. |
| CUDA out of memory | Lower `MAX_PIXELS` (786432 or 589824), set `QUANT='bnb4'` or `OFFLOAD='model'`, then restart the runtime. |
| Very slow on T4 | `DTYPE='fp16'`, `MAX_PIXELS=786432`, steps 8–12. |
| Download fails | Free disk: `!df -h /content`. Qwen needs ~45 GB free. Retry; HF resumes. |
| Tunnel URL dead | Re-run cell 6; quick tunnels are temporary. |
| Editor closes | Runtime reset — everything is in RAM, so images and history are gone. |

**See the server log:** `!tail -n 40 /tmp/herogod-server.log`

**Model licence:** Qwen-Image-Edit-2511 is Apache-2.0; the MCNL LoRA is OpenRAIL++. Read both model cards.


## Stop

In [ ]:
for name in ("tunnel_process", "server_process"):
    proc = globals().get(name)
    if proc and proc.poll() is None:
        proc.terminate()
        print("Stopped", name)
